# Assignment 2 — Hyperparameter Tuning

**Purpose:** Improve one LSTM through causal features, optimization recipes, and paired rolling selection.

This notebook is an executable evidence view over the shared local pipeline. The
command-line runner and notebook call the same stage implementation, preventing code
drift. Completed artifacts are reused; no Google Colab or Google Drive runtime is used.


In [1]:
# Local WSL project bootstrap. Dependency installation belongs to the
# locked project environment and is intentionally not repeated in notebooks.
from pathlib import Path
import json
import os
import platform
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [candidate, *candidate.parents] if (path / "shared").exists()),
    candidate,
)
if not (PROJECT_ROOT / "shared").exists():
    raise FileNotFoundError(
        "Open this notebook from the local odl-assignment project root."
    )
os.environ["ODL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import PATHS
from shared.runtime import RuntimeOptions
from shared.workflow import run_stage

OPTIONS = RuntimeOptions.from_environment()
print("Project root:", PROJECT_ROOT)
print("Run mode:", OPTIONS.mode)
print("Device policy:", OPTIONS.device)
print("WSL distribution:", os.getenv("WSL_DISTRO_NAME", "not detected"))


I0000 00:00:1786699779.978185   26846 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /home/nicholas/code/odl-assignment
Run mode: full
Device policy: gpu
WSL distribution: Ubuntu-24.04


I0000 00:00:1786699781.121150   26846 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Run the shared stage or load its verified completed result.
STAGE_SUMMARY = run_stage("tune", options=OPTIONS)
STAGE_SUMMARY


{'architecture': 'lstm',
 'selected_feature_policy': 'base',
 'selected_recipe': 'recipe_00',
 'recipe_validation_fits': 36,
 'candidate_count': 12,
 'validation_fits': 72,
 'selected_candidate': 'candidate_07',
 'formulation': 'residual',
 'validation_mae': 9.00888959846239,
 'validation_gate_passed': True,
 'validation_gate_bypassed_for_smoke': False,
 'epoch_budget': 9,
 'training_seconds': 312.4830053820042,
 'run_mode': 'full'}

In [3]:
# Record the exact runtime used for this output when preflight evidence exists.
environment_path = PATHS.environment_artifacts / f"environment_{OPTIONS.mode}.json"
ENVIRONMENT = (
    json.loads(environment_path.read_text(encoding="utf-8"))
    if environment_path.exists()
    else {"status": "preflight evidence not yet generated"}
)
ENVIRONMENT


{'captured_at_utc': '2026-08-14T09:29:07.880963+00:00',
 'project_root': '/home/nicholas/code/odl-assignment',
 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'system': 'Linux',
 'release': '6.6.87.2-microsoft-standard-WSL2',
 'python': '3.12.3',
 'wsl_distribution': 'Ubuntu-24.04',
 'device': 'gpu',
 'tensorflow_gpu_devices': ['/physical_device:GPU:0'],
 'batch_size': 256,
 'mixed_precision': False,
 'nvidia_smi': {'available': True,
  'name': 'NVIDIA GeForce RTX 3050 Laptop GPU',
  'driver_version': '591.74',
  'memory.total': '4096',
  'memory.used': '167',
  'utilization.gpu': '3',
  'temperature.gpu': '62',
  'power.draw': '18.49'},
 'packages': {'tensorflow': '2.21.0',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyarrow': '25.0.0',
  'scikit-learn': '1.9.0',
  'matplotlib': '3.11.1',
  'seaborn': '0.13.2'},
 'git': {'available': True,
  'revision': '4e8a886f8c5ec31e376ce1fd0a256c49b441fea9',
  'dirty': True},
 'experiment_config': {'methodology_versi

## Staged, focused tuning evidence

**Decision.** First compare six preregistered loss/optimizer recipes using candidate
13's promising residual architecture, then evaluate 12 deterministic local candidates
across three rolling folds and seeds 42/52. The local search contains the basic and
candidate-13 anchors plus ten balanced configurations. Random search is efficient when
only some dimensions strongly affect performance
([Bergstra and Bengio](https://www.jmlr.org/papers/v13/bergstra12a.html)). **Alternatives.**
A broad second global search would spend scarce GPU time in regions already shown to be
weak. **Trade-off.** The focused design costs 108 recipe/search fits after the 18-fit
feature experiment, but every decision remains based on rolling MAE. The frozen
benchmark is never read during selection.


In [4]:
import pandas as pd

suffix = OPTIONS.artifact_suffix
recipe_trials = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_results{suffix}.csv"
)
recipe_summary = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_summary{suffix}.csv"
)
recipe_audit = pd.read_csv(
    PATHS.assignment2_tables / f"optimization_recipe_paired_audit{suffix}.csv"
)
trials = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_search_results{suffix}.csv"
)
summary = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_search_summary{suffix}.csv"
)
paired_audit = pd.read_csv(
    PATHS.assignment2_tables / f"focused_lstm_paired_audit{suffix}.csv"
)
best_hyperparameters = json.loads(
    (PATHS.models / f"best_hyperparameters{suffix}.json").read_text(encoding="utf-8")
)
display(recipe_summary)
display(recipe_audit)
display(recipe_trials)
display(summary)
display(paired_audit)
display(trials)
display(best_hyperparameters)


,candidate_id,mean_mae,std_mae,folds,mean_seed_std_mae,min_seeds_per_fold,max_seeds_per_fold,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae,symmetric_loss_rank,optimizer_rank,mechanism_rank
0,recipe_00,9.022029,0.569800,3,0.033901,2,2,6,7.500000,7.5,226945,2,residual,0.328974,0,0,0
1,recipe_03,9.025137,0.575487,3,0.023457,2,2,6,9.666667,9.5,226945,2,residual,0.332257,0,0,2
2,recipe_02,9.033145,0.560030,3,0.018082,2,2,6,8.833333,9.0,226945,2,residual,0.323334,0,0,2
3,recipe_05,9.033145,0.560030,3,0.018082,2,2,6,8.833333,9.0,226945,2,residual,0.323334,0,1,3
4,recipe_01,9.039123,0.570979,3,0.021547,2,2,6,8.500000,9.0,226945,2,residual,0.329655,0,0,2
5,recipe_04,9.167591,0.499615,3,0.099826,2,2,6,11.833333,9.5,226945,2,residual,0.288453,1,0,2


,candidate_id,raw_winner,mean_paired_degradation,paired_standard_error,paired_one_se_eligible,paired_folds
0,recipe_00,recipe_00,0.000000,0.000000,True,3
1,recipe_03,recipe_00,0.003108,0.017978,True,3
2,recipe_02,recipe_00,0.011115,0.017416,True,3
3,recipe_05,recipe_00,0.011115,0.017416,True,3
4,recipe_01,recipe_00,0.017094,0.005107,False,3
5,recipe_04,recipe_00,0.145562,0.051141,False,3


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,loss_name,...,source_fingerprint,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.660940,17.885161,0.938623,19.184767,9,15,226945,164.471304
1,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.615709,17.797688,0.939222,19.185743,9,15,226945,165.148275
2,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,8.945010,17.233745,0.966899,18.224148,6,12,226945,175.975813
3,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,8.881790,17.006788,0.967766,18.177369,6,12,226945,166.089607
4,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,59962,8.496674,17.065648,0.935613,20.456965,8,14,226945,241.136275
5,recipe_00,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,59962,8.532053,17.108826,0.935287,20.573928,7,13,226945,203.617412
6,recipe_01,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.659646,17.949442,0.938181,19.200190,9,15,226945,191.978643
7,recipe_01,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.646499,17.897884,0.938536,19.331493,9,15,226945,189.677265
8,recipe_01,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,8.978558,17.302699,0.966634,18.339486,6,12,226945,186.463072
9,recipe_01,residual,128,64,0.2,0.00056,0.0001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,8.901995,17.000539,0.967789,18.352612,6,12,226945,180.637457


,candidate_id,mean_mae,std_mae,folds,mean_seed_std_mae,min_seeds_per_fold,max_seeds_per_fold,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae
0,candidate_07,9.008890,0.590961,3,0.025677,2,2,6,9.333333,9.0,62273,2,residual,0.341192
1,candidate_01,9.022029,0.569800,3,0.033901,2,2,6,7.500000,7.5,226945,2,residual,0.328974
2,candidate_05,9.023239,0.578210,3,0.025310,2,2,6,10.833333,11.0,62273,2,residual,0.333829
3,candidate_06,9.032605,0.609753,3,0.025679,2,2,6,5.833333,6.0,95361,1,residual,0.352041
4,candidate_08,9.034595,0.565455,3,0.013887,2,2,6,9.833333,10.5,133377,2,residual,0.326466
5,candidate_10,9.058648,0.615449,3,0.032018,2,2,6,9.333333,11.0,29249,1,residual,0.355330
6,candidate_03,9.061720,0.579864,3,0.013404,2,2,6,9.833333,9.5,226945,2,residual,0.334784
7,candidate_09,9.091134,0.585256,3,0.029342,2,2,6,11.833333,12.5,31361,1,direct,0.337898
8,candidate_00,9.095029,0.608228,3,0.027308,2,2,6,12.500000,12.0,29249,1,direct,0.351160
9,candidate_04,9.102689,0.625656,3,0.019755,2,2,6,7.666667,8.5,91201,1,residual,0.361223


,candidate_id,raw_winner,mean_paired_degradation,paired_standard_error,paired_one_se_eligible,paired_folds
0,candidate_07,candidate_07,0.000000,0.000000,True,3
1,candidate_01,candidate_07,0.013140,0.019093,True,3
2,candidate_05,candidate_07,0.014349,0.011928,False,3
3,candidate_06,candidate_07,0.023715,0.013673,False,3
4,candidate_08,candidate_07,0.025705,0.015777,False,3
5,candidate_10,candidate_07,0.049759,0.019209,False,3
6,candidate_03,candidate_07,0.052831,0.018175,False,3
7,candidate_09,candidate_07,0.082245,0.034088,False,3
8,candidate_00,candidate_07,0.086140,0.078014,False,3
9,candidate_04,candidate_07,0.093799,0.022176,False,3


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,loss_name,...,source_fingerprint,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.678562,17.838595,0.938942,19.040557,17,23,29249,190.917642
1,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.650165,17.919142,0.938390,18.788472,11,17,29249,140.459544
2,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,9.198754,18.047799,0.963699,18.154956,7,13,29249,133.140575
3,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,9.134248,17.769561,0.964809,17.924238,13,19,29249,199.545746
4,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,59962,8.442746,17.321198,0.933670,19.970693,11,17,29249,211.041067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,candidate_11,direct,96,64,0.1,0.000351,0.00001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51203,9.781317,18.190799,0.936507,19.170610,9,15,133377,152.791704
68,candidate_11,direct,96,64,0.1,0.000351,0.00001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,9.264957,18.244683,0.962902,17.801539,13,19,133377,290.717175
69,candidate_11,direct,96,64,0.1,0.000351,0.00001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,51790,9.280541,17.966649,0.964024,17.890943,22,28,133377,381.687087
70,candidate_11,direct,96,64,0.1,0.000351,0.00001,2,5.0,huber,...,c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab3...,59962,8.461003,17.765696,0.930222,19.846903,14,20,133377,345.668791


{'architecture': 'lstm',
 'objective': 'mean_rolling_validation_mae',
 'values': {'candidate_id': 'candidate_07',
  'formulation': 'residual',
  'units': 64,
  'dense_units': 32,
  'dropout': 0.3,
  'learning_rate': 0.0008691479198246962,
  'l2_strength': 1e-05,
  'num_layers': 2,
  'huber_delta': 5.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': None,
  'use_ema': False,
  'ema_momentum': 0.99,
  'batch_size': 128,
  'source': 'focused_local_search'},
 'selected_feature_policy': 'base',
 'selected_optimization_recipe': {'candidate_id': 'recipe_00',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 5.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': None,
  'use_ema': False,
  'ema_momentum': 0.99,
  'batch_size': 256,
  'sourc

## Final validation selection and fixed official model

**Decision.** Average seeds within each fold and compare every configuration directly
with the raw winner. Simplicity is used only when the mean paired degradation is no
greater than its paired fold-level standard error. A non-basic configuration must beat
the official basic LSTM on mean MAE and at least two folds before the benchmark can be
opened. The selected configuration is trained once with seed 42 for the median
validation-derived epoch budget. **Trade-off.** The gate can reject an apparent small
gain, but prevents a benchmark-driven retry and makes the final claim auditable.


In [5]:
final_selection = json.loads(
    (PATHS.models / f"final_model_selection{suffix}.json").read_text(encoding="utf-8")
)
validation_gate = json.loads(
    (PATHS.models / f"focused_lstm_validation_gate{suffix}.json").read_text(encoding="utf-8")
)
display(final_selection)
display(validation_gate)
print("Rolling tuning and single-model selection complete.")


{'architecture': 'lstm',
 'selected_candidate': {'candidate_id': 'candidate_07',
  'formulation': 'residual',
  'units': 64,
  'dense_units': 32,
  'dropout': 0.3,
  'learning_rate': 0.0008691479198246962,
  'l2_strength': 1e-05,
  'num_layers': 2,
  'huber_delta': 5.0,
  'loss_name': 'huber',
  'pinball_tau': 0.5,
  'optimizer_name': 'adam',
  'weight_decay': 0.0,
  'clipnorm': None,
  'use_ema': False,
  'ema_momentum': 0.99,
  'batch_size': 128,
  'source': 'focused_local_search'},
 'selected_seed': 42,
 'epoch_budget': 9,
 'selection_metric': 'mean_rolling_validation_mae',
 'mean_validation_mae': 9.00888959846239,
 'validation_standard_error': 0.3411915691925724,
 'selected_feature_policy': 'base',
 'selected_optimization_recipe': {'candidate_id': 'recipe_00',
  'formulation': 'residual',
  'units': 128,
  'dense_units': 64,
  'dropout': 0.2,
  'learning_rate': 0.000559702206464,
  'l2_strength': 0.0001,
  'num_layers': 2,
  'huber_delta': 5.0,
  'loss_name': 'huber',
  'pinball_ta

{'passed': True,
 'bypassed_for_smoke': False,
 'checks': {'non_basic_configuration': True,
  'lower_mean_mae_than_basic_lstm': True,
  'wins_required_fold_averages': True,
  'complete_search': True},
 'selected_candidate': 'candidate_07',
 'selected_source': 'focused_local_search',
 'selected_mean_mae': 9.00888959846239,
 'basic_lstm_mean_mae': 9.114681824203016,
 'fold_wins': 3,
 'required_fold_wins': 2,
 'paired_fold_metrics': [{'fold': 'fold_1',
   'tuned_mae': 9.636997234425763,
   'basic_mae': 9.677919211594702},
  {'fold': 'fold_2',
   'tuned_mae': 8.925802572244475,
   'basic_mae': 9.174978789205507},
  {'fold': 'fold_3',
   'tuned_mae': 8.463868988716937,
   'basic_mae': 8.49114747180884}],
 'source_fingerprint': 'c9c4e3812d15bff6e6cf7737963fb968a0f31ba6130ab364434d62d892fdac46'}

Rolling tuning and single-model selection complete.
